# Claude Agent SDK: 子agents 和会话存储

Claude Agent 的 SKD 是 Claude Code harness 的库形式。内置工具、上下文隔离的子agents、钩子、W3C 轨迹传播、会话存储持久。Claude Managed Agents则是面向长时异步工作的托管替代。

## 问题描述

一个原生的LLM API 给你一轮往返。一个生产级的agent需要工具调用、MCP服务、声明周期hooks、子agent派生、会话持久、trace 传播。Claude Agent SDK把这种形态做一个库开放出来————也是Claude Code 用的harness，暴露出来给自定义agent。

## 基本概念

### Client SDK vs Agent SDK

- **Client SDK（anthropic）**。 原生的消息API，你自己掌控循环、工具和状态。

- **Agent SDK（cluade-agent-sdk）**。 内置工具执行、MCP连接、钩子、子agent 派生，会话存储。Claude Code 循环对应的库。

### 内置工具

SKD 有10余种内置工具：文件读写、shell、grep... 也支持通过标准schema 模式注册自定义工具。

### 子agents

两种目标：
1. 并行。 并发的跑独立任务。
2. 上下文隔离。 子agents使用各自的上下文窗口，只有结果才会回到编排器。因此编排器的token预算得以保存。

### 会话存储

* `append(session_id, message)`
* `load(session_id)`
* `list_sessions()`
* `delete(session_id)`
* `list_subkeys(session_id)`

### Hooks

* `Pre/PostToolUse`
* `SessionStart/End`
* `UserPromptSubmmit`
* `PreCompat`
* `Stop`
* `Notification`

### 什么时候这种模式失效

- 子agent过渡使用。 100个小任务创建了100个子agents，使用批处理。
- Hook蔓延。 到处都是钩子
- 会话膨胀。 会话越攒越多，体积膨胀。使用`list_sessions` 然后移除过期内容。

# 开始编码

对应本章核心：**子 agent（并行 + 上下文隔离）**、**SessionStore**、**生命周期 Hooks**、**W3C traceparent 传播**、**内置/自定义工具**。  
先用玩具 harness 跑通编排器派生子 agent；再用 **LangChain + DeepSeek** 做真实子任务（不硬凑 PyTorch）。


## 1. 教学玩具：Claude Agent 风格 Harness

- **SessionStore**：`append/load/list/delete/list_subkeys`；可清过期防膨胀。
- **Hooks**：`PreToolUse` / `PostToolUse` / `SessionStart` / `SessionEnd` / `UserPromptSubmit`。
- **SubAgent**：独立上下文；只把 `result` 回传编排器。
- **TraceContext**：生成/解析 W3C `traceparent`，子 agent 继承同一 `trace_id`。


In [ ]:
from __future__ import annotations

import re
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

HookName = Literal[
    "PreToolUse",
    "PostToolUse",
    "SessionStart",
    "SessionEnd",
    "UserPromptSubmit",
    "Stop",
    "Notification",
]
HookFn = Callable[[dict[str, Any]], None]
ToolFn = Callable[[dict[str, Any]], str]


@dataclass
class TraceContext:
    """W3C Trace Context（简化：只实现 traceparent）。"""

    trace_id: str
    span_id: str
    flags: str = "01"

    @staticmethod
    def new() -> TraceContext:
        return TraceContext(trace_id=uuid.uuid4().hex, span_id=uuid.uuid4().hex[:16])

    def child(self) -> TraceContext:
        """同 trace_id，新 span_id（传播到子 agent）。"""
        return TraceContext(trace_id=self.trace_id, span_id=uuid.uuid4().hex[:16], flags=self.flags)

    def to_traceparent(self) -> str:
        """
        Returns:
            header: ``00-<trace_id>-<span_id>-<flags>``。
        """
        return f"00-{self.trace_id}-{self.span_id}-{self.flags}"

    @staticmethod
    def from_traceparent(header: str) -> TraceContext:
        """
        Args:
            header: W3C traceparent。

        Returns:
            ctx: 解析结果。
        """
        parts = header.strip().split("-")
        if len(parts) != 4 or parts[0] != "00":
            raise ValueError(f"bad traceparent: {header}")
        return TraceContext(trace_id=parts[1], span_id=parts[2], flags=parts[3])


@dataclass
class SessionStore:
    """会话持久（玩具：进程内 dict）。"""

    _data: dict[str, dict[str, Any]] = field(default_factory=dict)

    def append(self, session_id: str, message: dict[str, Any]) -> None:
        """
        Args:
            session_id: 会话 ID。
            message: 一条消息（可含 subkey）。
        """
        slot = self._data.setdefault(
            session_id, {"messages": [], "subkeys": set(), "updated_at": time.time()}
        )
        slot["messages"].append(message)
        if "subkey" in message:
            slot["subkeys"].add(message["subkey"])
        slot["updated_at"] = time.time()

    def load(self, session_id: str) -> list[dict[str, Any]]:
        return list(self._data.get(session_id, {}).get("messages", []))

    def list_sessions(self) -> list[str]:
        return sorted(self._data)

    def delete(self, session_id: str) -> bool:
        return self._data.pop(session_id, None) is not None

    def list_subkeys(self, session_id: str) -> list[str]:
        return sorted(self._data.get(session_id, {}).get("subkeys", set()))

    def purge_older_than(self, seconds: float) -> list[str]:
        """
        会话膨胀修法：删过期会话。

        Returns:
            removed: 被删 session_id。
        """
        now = time.time()
        dead = [sid for sid, v in self._data.items() if now - v.get("updated_at", now) > seconds]
        for sid in dead:
            self.delete(sid)
        return dead


class HookBus:
    """生命周期钩子（控制数量，避免 hook 蔓延）。"""

    def __init__(self, *, max_per_event: int = 3) -> None:
        self._hooks: dict[str, list[HookFn]] = {}
        self.max_per_event = max_per_event
        self.log: list[str] = []

    def on(self, name: HookName, fn: HookFn) -> None:
        """
        Args:
            name: 钩子名。
            fn: 回调。
        """
        bucket = self._hooks.setdefault(name, [])
        if len(bucket) >= self.max_per_event:
            raise RuntimeError(f"hook sprawl blocked: {name} already has {self.max_per_event}")
        bucket.append(fn)

    def emit(self, name: HookName, payload: dict[str, Any]) -> None:
        self.log.append(name)
        for fn in self._hooks.get(name, []):
            fn(payload)


@dataclass
class Tool:
    name: str
    description: str
    fn: ToolFn


def builtin_tools() -> dict[str, Tool]:
    """少量内置工具示意（非完整 Claude 工具集）。"""

    def read_file(args: dict[str, Any]) -> str:
        return f"<file:{args.get('path', '')}> stub content"

    def grep(args: dict[str, Any]) -> str:
        return f"grep({args.get('pattern')}) -> 0 hits (stub)"

    return {
        "read_file": Tool("read_file", "Read a file", read_file),
        "grep": Tool("grep", "Search files", grep),
    }


@dataclass
class SubAgentResult:
    name: str
    result: str
    context_tokens: int  # 子上下文用量（不回灌编排器）
    traceparent: str


@dataclass
class SubAgent:
    """上下文隔离的子 agent：私有 messages，只返回 result。"""

    name: str
    system: str
    worker: Callable[[str, list[dict[str, str]]], str]
    messages: list[dict[str, str]] = field(default_factory=list)

    def run(self, task: str, *, parent_trace: TraceContext) -> SubAgentResult:
        """
        Args:
            task: 子任务。
            parent_trace: 父上下文（传播 trace_id）。

        Returns:
            result: 仅结果回编排器。
        """
        child_trace = parent_trace.child()
        self.messages = [
            {"role": "system", "content": self.system},
            {"role": "user", "content": task},
        ]
        out = self.worker(task, self.messages)
        self.messages.append({"role": "assistant", "content": out})
        # 粗算 token：字符/4；编排器看不到这些 messages
        tokens = sum(len(m["content"]) for m in self.messages) // 4
        return SubAgentResult(self.name, out, tokens, child_trace.to_traceparent())


class AgentHarness:
    """Agent SDK 风格：工具 + hooks + session + 子 agent 派生。"""

    def __init__(
        self,
        *,
        store: SessionStore | None = None,
        hooks: HookBus | None = None,
        tools: dict[str, Tool] | None = None,
        max_subagents: int = 8,
    ) -> None:
        self.store = store or SessionStore()
        self.hooks = hooks or HookBus()
        self.tools = tools or builtin_tools()
        self.max_subagents = max_subagents
        self.orchestrator_context: list[str] = []  # 仅摘要，省 token

    def register_tool(self, tool: Tool) -> None:
        self.tools[tool.name] = tool

    def call_tool(self, name: str, args: dict[str, Any], *, session_id: str, trace: TraceContext) -> str:
        """
        Args:
            name: 工具名。
            args: 参数。
            session_id: 会话。
            trace: 当前 trace。

        Returns:
            output: 工具输出。
        """
        if name not in self.tools:
            raise KeyError(name)
        self.hooks.emit(
            "PreToolUse",
            {"tool": name, "args": args, "traceparent": trace.to_traceparent()},
        )
        out = self.tools[name].fn(args)
        self.hooks.emit("PostToolUse", {"tool": name, "output": out[:200]})
        self.store.append(
            session_id,
            {"role": "tool", "name": name, "content": out, "subkey": f"tool:{name}"},
        )
        return out

    def spawn_subagents(
        self,
        specs: list[tuple[str, str, Callable[[str, list[dict[str, str]]], str]]],
        tasks: list[str],
        *,
        parent_trace: TraceContext,
        parallel: bool = True,
    ) -> list[SubAgentResult]:
        """
        Args:
            specs: ``(name, system, worker)``。
            tasks: 与 specs 对齐的任务；过长应批处理而非一人一 agent。
            parent_trace: 父 trace。
            parallel: 是否并发。

        Returns:
            results: 子结果列表。
        """
        if len(specs) > self.max_subagents:
            raise RuntimeError(
                f"too many subagents ({len(specs)}>{self.max_subagents}); batch tasks instead"
            )
        agents = [SubAgent(n, sys, w) for (n, sys, w) in specs]

        def _one(i: int) -> SubAgentResult:
            return agents[i].run(tasks[i], parent_trace=parent_trace)

        if not parallel:
            return [_one(i) for i in range(len(agents))]
        outs: list[SubAgentResult | None] = [None] * len(agents)
        with ThreadPoolExecutor(max_workers=len(agents)) as pool:
            futs = {pool.submit(_one, i): i for i in range(len(agents))}
            for fut in as_completed(futs):
                outs[futs[fut]] = fut.result()
        return [o for o in outs if o is not None]

    def run(
        self,
        session_id: str,
        prompt: str,
        *,
        sub_specs: list[tuple[str, str, Callable[[str, list[dict[str, str]]], str]]] | None = None,
        sub_tasks: list[str] | None = None,
        trace: TraceContext | None = None,
    ) -> dict[str, Any]:
        """
        编排一轮：SessionStart → Prompt →（可选）并行子 agent → 汇总。

        Returns:
            payload: 最终输出 + trace + 子结果摘要。
        """
        trace = trace or TraceContext.new()
        self.hooks.emit("SessionStart", {"session_id": session_id, "traceparent": trace.to_traceparent()})
        self.hooks.emit("UserPromptSubmit", {"prompt": prompt})
        self.store.append(session_id, {"role": "user", "content": prompt})

        sub_results: list[SubAgentResult] = []
        if sub_specs and sub_tasks:
            sub_results = self.spawn_subagents(sub_specs, sub_tasks, parent_trace=trace, parallel=True)
            # 关键：只把 result 写入编排器上下文，不把子 messages 整坨塞回
            for r in sub_results:
                summary = f"[{r.name}] {r.result}"
                self.orchestrator_context.append(summary)
                self.store.append(
                    session_id,
                    {
                        "role": "subagent",
                        "subkey": f"sub:{r.name}",
                        "content": r.result,
                        "traceparent": r.traceparent,
                        "child_tokens": r.context_tokens,
                    },
                )

        final = " | ".join(self.orchestrator_context[-len(sub_results) :] or [f"echo:{prompt}"])
        self.store.append(session_id, {"role": "assistant", "content": final})
        self.hooks.emit("SessionEnd", {"session_id": session_id})
        self.hooks.emit("Stop", {"reason": "completed"})
        return {
            "output": final,
            "traceparent": trace.to_traceparent(),
            "subagents": [
                {
                    "name": r.name,
                    "result": r.result,
                    "context_tokens": r.context_tokens,
                    "traceparent": r.traceparent,
                }
                for r in sub_results
            ],
            "orchestrator_tokens": sum(len(x) for x in self.orchestrator_context) // 4,
        }


print("Claude Agent harness ready | subagents + session + hooks + W3C trace")


## 2. 玩具示例：隔离、并行、传播、会话清理


In [ ]:
def demo_claude_agent_toy() -> None:
    """断言上下文隔离、并行、trace 传播、hooks 上限、会话 purge。"""
    hooks = HookBus(max_per_event=3)
    hook_payloads: list[str] = []
    hooks.on("PreToolUse", lambda p: hook_payloads.append("pre:" + p["tool"]))
    hooks.on("PostToolUse", lambda p: hook_payloads.append("post"))
    hooks.on("SessionStart", lambda p: hook_payloads.append("start"))

    # hook 蔓延被挡住
    hooks.on("SessionStart", lambda p: None)
    hooks.on("SessionStart", lambda p: None)
    try:
        hooks.on("SessionStart", lambda p: None)
        raise AssertionError("expected hook sprawl block")
    except RuntimeError as e:
        assert "hook sprawl" in str(e)
        print("hook sprawl blocked:", e)

    store = SessionStore()
    harness = AgentHarness(store=store, hooks=hooks, max_subagents=4)

    # 自定义工具
    harness.register_tool(Tool("add", "add two ints", lambda a: str(int(a["x"]) + int(a["y"]))))
    parent = TraceContext.new()
    tool_out = harness.call_tool("add", {"x": 2, "y": 3}, session_id="s1", trace=parent)
    assert tool_out == "5"
    assert "pre:add" in hook_payloads

    def worker_factory(tag: str) -> Callable[[str, list[dict[str, str]]], str]:
        def _w(task: str, messages: list[dict[str, str]]) -> str:
            # 子上下文很长，但不应进入编排器
            pad = "detail " * 50
            return f"{tag}:{task}:{len(messages)}:{pad[:12]}"

        return _w

    specs = [
        ("research", "researcher", worker_factory("R")),
        ("draft", "writer", worker_factory("D")),
    ]
    tasks = ["topic-A", "topic-A-outline"]
    result = harness.run("s1", "write about A", sub_specs=specs, sub_tasks=tasks, trace=parent)

    # 同一 trace_id 传播
    parent_tp = TraceContext.from_traceparent(result["traceparent"])
    child_tps = [TraceContext.from_traceparent(s["traceparent"]) for s in result["subagents"]]
    assert all(c.trace_id == parent_tp.trace_id for c in child_tps)
    assert all(c.span_id != parent_tp.span_id for c in child_tps)
    print("W3C propagation ok:", parent_tp.trace_id[:8], "→", [c.span_id[:6] for c in child_tps])

    # 上下文隔离：子 agent token 远大于编排器摘要
    child_tokens = sum(s["context_tokens"] for s in result["subagents"])
    assert child_tokens > result["orchestrator_tokens"]
    # 编排器上下文不应含 "detail detail" 长垫
    assert all("detail detail" not in x for x in harness.orchestrator_context)
    print("isolation ok: child_tokens=", child_tokens, "orch=", result["orchestrator_tokens"])

    # 子 agent 过多
    try:
        too_many = [(f"a{i}", "s", worker_factory("x")) for i in range(9)]
        harness.spawn_subagents(too_many, ["t"] * 9, parent_trace=parent)
        raise AssertionError("expected too many subagents")
    except RuntimeError as e:
        assert "batch" in str(e)
        print("subagent storm blocked:", e)

    # 会话 API
    assert "s1" in store.list_sessions()
    assert "sub:research" in store.list_subkeys("s1")
    store.append("old", {"role": "user", "content": "x"})
    store._data["old"]["updated_at"] = time.time() - 10_000
    removed = store.purge_older_than(3600)
    assert "old" in removed and "old" not in store.list_sessions()
    print("session purge:", removed)

    print("TOY DEMO OK")


demo_claude_agent_toy()


## 3. 生产级：子 Agent + LangChain / DeepSeek

编排器派生 research/writer 两个 LLM 子 agent（上下文隔离）；工具暴露 `run_harness` / `session_ops` / `get_trace_header`。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Callable, Literal

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_STORE = SessionStore()
PROD_HOOKS = HookBus(max_per_event=3)
PROD_HARNESS = AgentHarness(store=PROD_STORE, hooks=PROD_HOOKS, max_subagents=4)
LAST_RUN: dict[str, Any] = {}


def get_llm(*, temperature: float = 0.2) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def make_llm_worker(system: str) -> Callable[[str, list[dict[str, str]]], str]:
    """
    Args:
        system: 子 agent 系统提示（保持短）。

    Returns:
        worker: SubAgent.worker。
    """

    def _worker(task: str, messages: list[dict[str, str]]) -> str:
        # 子 agent 只用自己的 messages，不看编排器全文
        prompt = f"{system}\n\nTask: {task}\nReply in Chinese, <=60 chars."
        return str(get_llm().invoke(prompt).content).strip()

    return _worker


def reset_prod() -> None:
    """重建生产 harness。"""
    global PROD_STORE, PROD_HOOKS, PROD_HARNESS, LAST_RUN
    PROD_STORE = SessionStore()
    PROD_HOOKS = HookBus(max_per_event=3)
    PROD_HARNESS = AgentHarness(store=PROD_STORE, hooks=PROD_HOOKS, max_subagents=4)
    LAST_RUN = {}


class RunHarnessArgs(BaseModel):
    session_id: str = Field(default="demo")
    prompt: str = Field(description="Orchestrator user prompt")
    topic: str = Field(description="Topic for research/writer subagents")


class SessionOpsArgs(BaseModel):
    op: Literal["list", "load", "delete", "subkeys", "purge"] = "list"
    session_id: str = "demo"
    older_than_sec: float = 3600


class EmptyArgs(BaseModel):
    pass


def run_harness_impl(session_id: str, prompt: str, topic: str) -> str:
    """
    Returns:
        json: 编排结果（含子 agent 摘要与 traceparent）。
    """
    global LAST_RUN
    specs = [
        ("research", "You are a researcher. Give 2 short facts.", make_llm_worker("researcher")),
        ("writer", "You are a writer. One tight sentence from facts.", make_llm_worker("writer")),
    ]
    tasks = [f"Research: {topic}", f"Draft from research about: {topic}"]
    LAST_RUN = PROD_HARNESS.run(session_id, prompt, sub_specs=specs, sub_tasks=tasks)
    return json.dumps(LAST_RUN, ensure_ascii=False)


def session_ops_impl(op: str, session_id: str, older_than_sec: float = 3600) -> str:
    """
    Returns:
        json: 会话存储操作结果。
    """
    if op == "list":
        return json.dumps({"sessions": PROD_STORE.list_sessions()})
    if op == "load":
        return json.dumps({"messages": PROD_STORE.load(session_id)}, ensure_ascii=False)
    if op == "delete":
        return json.dumps({"deleted": PROD_STORE.delete(session_id)})
    if op == "subkeys":
        return json.dumps({"subkeys": PROD_STORE.list_subkeys(session_id)})
    if op == "purge":
        return json.dumps({"removed": PROD_STORE.purge_older_than(older_than_sec)})
    return json.dumps({"error": f"unknown op {op}"})


def get_last_trace_impl() -> str:
    """
    Returns:
        json: 父/子 traceparent，验证传播。
    """
    if not LAST_RUN:
        return json.dumps({"error": "no run yet"})
    parent = LAST_RUN.get("traceparent")
    children = [s.get("traceparent") for s in LAST_RUN.get("subagents", [])]
    parent_id = TraceContext.from_traceparent(parent).trace_id if parent else None
    child_ids = [TraceContext.from_traceparent(c).trace_id for c in children if c]
    return json.dumps(
        {
            "parent": parent,
            "children": children,
            "same_trace": bool(parent_id) and all(t == parent_id for t in child_ids),
            "hooks": PROD_HOOKS.log[-12:],
        },
        ensure_ascii=False,
    )


def build_claude_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: harness 控制面。
    """

    def _reset(**kwargs: Any) -> str:
        reset_prod()
        return json.dumps({"status": "reset"})

    def _run(**kwargs: Any) -> str:
        a = RunHarnessArgs(**kwargs)
        return run_harness_impl(a.session_id, a.prompt, a.topic)

    def _sess(**kwargs: Any) -> str:
        a = SessionOpsArgs(**kwargs)
        return session_ops_impl(a.op, a.session_id, a.older_than_sec)

    def _trace(**kwargs: Any) -> str:
        return get_last_trace_impl()

    return [
        StructuredTool.from_function(
            name="reset_harness",
            description="Reset session store, hooks, and harness.",
            func=_reset,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="run_harness",
            description="Run orchestrator with isolated research/writer subagents and W3C trace propagation.",
            func=_run,
            args_schema=RunHarnessArgs,
        ),
        StructuredTool.from_function(
            name="session_ops",
            description="Session store ops: list|load|delete|subkeys|purge.",
            func=_sess,
            args_schema=SessionOpsArgs,
        ),
        StructuredTool.from_function(
            name="get_last_trace",
            description="Show parent/child traceparent and whether trace_id propagated.",
            func=_trace,
            args_schema=EmptyArgs,
        ),
    ]


CLAUDE_TOOLS = build_claude_tools()


def build_control_agent() -> Any:
    """
    Returns:
        agent: 通过工具驱动 Claude-Agent 风格 harness。
    """
    system = (
        "You operate a Claude-Agent-style harness.\n"
        "Flow: reset_harness -> run_harness -> get_last_trace / session_ops.\n"
        "Explain subagent isolation and W3C propagation in Chinese."
    )
    return create_agent(get_llm(), CLAUDE_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 500 else str(m.content)[:500] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"Claude-Agent-style + LangChain ready | {MODEL}")


## 4. 生产示例：并行子 Agent + Trace 传播


In [ ]:
def demo_deepseek_claude_agent() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod()
    print("=== run harness ===")
    out = json.loads(
        run_harness_impl("prod1", "请研究并起草简介", "W3C trace propagation")
    )
    print(json.dumps(out, ensure_ascii=False, indent=2)[:900])
    assert out.get("output")
    assert len(out.get("subagents", [])) == 2
    child_tok = sum(s["context_tokens"] for s in out["subagents"])
    assert child_tok >= out.get("orchestrator_tokens", 0)

    trace = json.loads(get_last_trace_impl())
    print("trace:", trace)
    assert trace.get("same_trace") is True

    sess = json.loads(session_ops_impl("subkeys", "prod1"))
    assert any(k.startswith("sub:") for k in sess.get("subkeys", []))
    print("subkeys:", sess)

    print("=== control agent ===")
    try:
        agent = build_control_agent()
        result = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=(
                            "reset_harness，然后 run_harness 主题「子agent上下文隔离」，"
                            "再 get_last_trace，用中文说明 trace_id 是否传播成功。"
                        )
                    )
                ]
            }
        )
        print(format_agent_messages(result["messages"]))
        assert count_tool_calls(result["messages"]) >= 2
    except Exception as e:
        print(f"control agent skipped due to LLM error: {type(e).__name__}: {e}")
    print("PRODUCTION DEMO OK")


demo_deepseek_claude_agent()
